In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics as skm

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
class_names = list(iris.target_names)
print("Ознаки:", iris.feature_names)
print("Класи:", class_names)
print("Розмір:", X.shape, "| розподіл класів:", np.bincount(y))

Ознаки: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Класи: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Розмір: (150, 4) | розподіл класів: [50 50 50]


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("Тестова вибірка:", len(y_test), "| у кожному класі:", np.bincount(y_test))
print("y_true:", y_test)
print("y_pred:", y_pred)

Тестова вибірка: 45 | у кожному класі: [15 15 15]
y_true: [2 1 2 1 2 2 1 1 0 2 0 0 2 2 0 2 1 0 0 0 1 0 1 2 2 1 1 1 1 0 2 2 1 0 2 0 0
 0 0 1 1 0 2 2 1]
y_pred: [2 1 2 1 2 2 1 1 0 2 0 0 2 2 0 2 1 0 0 0 1 0 1 2 2 1 1 1 1 0 2 2 1 0 2 0 0
 0 0 1 1 0 1 2 1]


In [4]:
correct = int(np.sum(y_test == y_pred))
total = len(y_test)
accuracy = correct / total
print(f"Правильних: {correct} з {total}")
print(f"Accuracy = {correct}/{total} = {accuracy:.4f}")

Правильних: 44 з 45
Accuracy = 44/45 = 0.9778


In [5]:
def confusion_counts(y_true, y_pred, c):
    tp = int(np.sum((y_true == c) & (y_pred == c)))
    fp = int(np.sum((y_true != c) & (y_pred == c)))
    fn = int(np.sum((y_true == c) & (y_pred != c)))
    tn = int(np.sum((y_true != c) & (y_pred != c)))
    return tp, fp, fn, tn

counts = {}
print(f"{'клас':<12}{'TP':>4}{'FP':>4}{'FN':>4}{'TN':>4}{'сума':>6}")
for c, name in enumerate(class_names):
    tp, fp, fn, tn = confusion_counts(y_test, y_pred, c)
    counts[c] = (tp, fp, fn, tn)
    print(f"{name:<12}{tp:>4}{fp:>4}{fn:>4}{tn:>4}{tp+fp+fn+tn:>6}")

клас          TP  FP  FN  TN  сума
setosa        15   0   0  30    45
versicolor    15   1   0  29    45
virginica     14   0   1  30    45


In [7]:
def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_score(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

P, R, F = [], [], []
print(f"{'клас':<12}{'Precision':>11}{'Recall':>9}{'F1':>9}")
for c, name in enumerate(class_names):
    tp, fp, fn, tn = counts[c]
    p, r = precision(tp, fp), recall(tp, fn)
    f = f1_score(p, r)
    P.append(p); R.append(r); F.append(f)
    print(f"{name:<12}{p:>11.4f}{r:>9.4f}{f:>9.4f}")

print()
print(f"Macro:    P={np.mean(P):.4f}  R={np.mean(R):.4f}  F1={np.mean(F):.4f}")

клас          Precision   Recall       F1
setosa           1.0000   1.0000   1.0000
versicolor       0.9375   1.0000   0.9677
virginica        1.0000   0.9333   0.9655

Macro:    P=0.9792  R=0.9778  F1=0.9778


In [8]:
p_sk, r_sk, f_sk, _ = skm.precision_recall_fscore_support(y_test, y_pred, zero_division=0)
print("Accuracy збігається:", np.isclose(accuracy, skm.accuracy_score(y_test, y_pred)))
print("Precision збігається:", np.allclose(P, p_sk))
print("Recall збігається:   ", np.allclose(R, r_sk))
print("F1 збігається:       ", np.allclose(F, f_sk))
print("Матриця помилок збігається:", np.array_equal(cm, skm.confusion_matrix(y_test, y_pred)))
print()
print(skm.classification_report(y_test, y_pred, target_names=class_names, digits=4))

Accuracy збігається: True
Precision збігається: True
Recall збігається:    True
F1 збігається:        True
Матриця помилок збігається: True

              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        15
  versicolor     0.9375    1.0000    0.9677        15
   virginica     1.0000    0.9333    0.9655        15

    accuracy                         0.9778        45
   macro avg     0.9792    0.9778    0.9778        45
weighted avg     0.9792    0.9778    0.9778        45

